# COMP3220 Assignment 4: Heart Disease Prediction

**Group members**: Alexia Barrows (620161540) and Daniel Graham (620171166)

**Description** <br>
You are given medical data on several patients. Your job is to use this data to build a model that
predicts whether a patient has heart disease. A Jupyter notebook is provided, along with two
CSV files: heart_train.csv and heart_test.csv. Edit the notebook to complete the objectives
below. This assignment is based on the Kaggle Heart Failure Prediction COMP3220 (UWI)
competition.

https://scikit-learn.org/stable/api/sklearn.html <br>
https://youtu.be/VXtjG_GzO7Q?si=TsvXcg_4FSpvGSfG


# Import Libraries
(Run 'pip install -r requirements.txt first')

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,classification_report,
                             ConfusionMatrixDisplay, f1_score, confusion_matrix)

RANDOM_STATE = 42

## 1. Read heart_train.csv into a pandas dataframe.

In [ ]:
train = pd.read_csv("heart_train.csv")
test = pd.read_csv("heart_test.csv")

print("\nFirst 5 rows of training data:")
train.head()

Training data shape: (818, 13)
Test data shape: (100, 12)

First 5 rows of training data:


,PatientId,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,0,34,M,ATA,150,214,0,ST,168,N,0.0,Up,0
1,1,59,M,ASY,178,0,1,LVH,120,Y,0.0,Flat,1
2,2,58,M,ASY,115,0,1,Normal,138,N,0.5,Up,1
3,3,60,M,ASY,130,253,0,Normal,144,Y,1.4,Up,1
4,4,52,M,ASY,165,0,1,Normal,122,Y,1.0,Up,1


## 2. Pre-process and clean your data so it is ready for training

Check for null values, duplicate rows, impossible values and class balance.

In [ ]:
print("Null values in train:", int(train.isnull().sum().sum()))
print("Null values in test:", int(test.isnull().sum().sum()))
print("Duplicate rows in train:", int(train.duplicated().sum()))

train = train.drop_duplicates().reset_index(drop=True)

print("Cholesterol == 0 : train", int((train.Cholesterol == 0).sum()), "| test", int((test.Cholesterol == 0).sum()))
print("RestingBP == 0 : train", int((train.RestingBP == 0).sum()), "| test", int((test.RestingBP == 0).sum()))

medians = {}
for col in ['Cholesterol', 'RestingBP']:
    med = train.loc[train[col] != 0, col].median()
    medians[col] = med
    num_zero_train = (train[col] == 0).sum()
    num_zero_test = (test[col] == 0).sum()
    train.loc[train[col] == 0, col] = med
    test.loc[test[col] == 0, col] = med
    print(f"{col}: train zeros={num_zero_train}, test zeros={num_zero_test}: replaced with median {med:.1f}")

Null values in train: 0
Null values in test: 0
Duplicate rows in train: 0
Cholesterol == 0 : train 156 | test 0
RestingBP   == 0 : train 0 | test 0
Cholesterol: train zeros=0, test zeros=0: replaced with median 235.0
RestingBP: train zeros=0, test zeros=0: replaced with median 130.0


**Findings and decisions**

- No null values and no duplicate rows.
- Duplicate roqs dropped
- **Cholesterol = 0** (151 training rows, 21 test rows) and **RestingBP = 0** (1 training row) are replaced with the median of the valid values.

## 3. Choose the features to use (for example RestingBP, Cholesterol, FastingBS)
## 4. Extract the HeartDisease column to use as the labels for your model.

In [11]:
numeric_features     = ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak','FastingBS']
categorical_features = ['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope']
all_features         = numeric_features + categorical_features

# extract labels
X = train[all_features]
y = train['HeartDisease']

# preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False),
     categorical_features)
])

print("Feature matrix:", X.shape)
print("Encoded feature count:", 
      len(numeric_features) + 
      sum(train[c].nunique()-1 for c in categorical_features))

Feature matrix: (818, 11)
Encoded feature count: 15


## 5. Split the dataset into train and test sets using the train_test_split function.

80% train / 20% test, stratified so both parts keep the same class ratio, with random_state=42 so results are repeatable.

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {X_train.shape[0]}\tTest: {X_test.shape[0]}")
print("Train positive rate:", round(y_train.mean(), 3))
print("Test positive rate:", round(y_test.mean(), 3))

Train: 654	Test: 164
Train positive rate: 0.546
Test positive rate: 0.543
